# Stage 6 & 7: Algorithm 1 — Logistic Regression

**Owner:** Member A  
**Inputs:** `data/final/` — X_train, X_val, y_train, y_val  
**Outputs:** `models/logistic_regression_tuned.pkl`  

### Scope
This notebook implements the linear baseline model family for late delivery risk prediction.
It trains a Dummy Classifier baseline, fits a standard Logistic Regression, performs 5-Fold Stratified K-Fold hyperparameter tuning on `X_train`, evaluates performance on `X_val`, and exports the tuned model artifact.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report
)

FINAL_DIR = Path('../data/final')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()

print(f'X_train shape: {X_train.shape}, X_val shape: {X_val.shape}')


X_train shape: (67529, 67), X_val shape: (14470, 67)


In [2]:
# ── Dummy Classifier Baseline ──────────────────────────────────────────────
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_probs = dummy.predict_proba(X_val)[:, 1]
print(f'Dummy Val ROC-AUC : {roc_auc_score(y_val, dummy_probs):.4f}')
print(f'Dummy Val PR-AUC  : {average_precision_score(y_val, dummy_probs):.4f}')


Dummy Val ROC-AUC : 0.5000
Dummy Val PR-AUC  : 0.0534


In [3]:
# ── 5-Fold Stratified K-Fold Hyperparameter Tuning ───────────────────────
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'class_weight': ['balanced', None]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid, scoring='average_precision', cv=cv, n_jobs=-1
)
grid.fit(X_train, y_train)

print('Best Hyperparameters:', grid.best_params_)
print(f'Best Stratified CV PR-AUC: {grid.best_score_:.4f}')

best_lr = grid.best_estimator_
joblib.dump(best_lr, MODELS_DIR / 'logistic_regression_tuned.pkl')
print('Saved model artifact to models/logistic_regression_tuned.pkl')


C:\Users\SKY PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Hyperparameters: {'C': 0.1, 'class_weight': None, 'penalty': 'l2', 'solver': 'lbfgs'}
Best Stratified CV PR-AUC: 0.2204
Saved model artifact to models/logistic_regression_tuned.pkl
